In [2]:
import pickle
import numpy as np
# pairwise distance
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr

In [3]:
dict_path = f"/home/tristan/Research/Sp25/embedor/outputs/server_experiments/embedor_developmental_25k.pkl"
with open(dict_path, "rb") as f:
    data = pickle.load(f)

In [26]:
embedor_emb = data["embedor_emb"]
umap_emb = data["umap_emb"]
tsne_emb = data["tsne_emb"]
time = data["labels"]

# find nan indices in time and drop them from all embeddings
nan_indices = np.where(np.isnan(time))[0]
if len(nan_indices) > 0:
    print(f"Found {len(nan_indices)} NaN indices in time, dropping them from all embeddings.")
    embedor_emb = np.delete(embedor_emb, nan_indices, axis=0)
    umap_emb = np.delete(umap_emb, nan_indices, axis=0)
    tsne_emb = np.delete(tsne_emb, nan_indices, axis=0)
    time = np.delete(time, nan_indices)

# randomly subsample 10k points
n_samples = 5000
indices = np.random.choice(len(embedor_emb), n_samples, replace=False)
embedor_emb_subsampled = embedor_emb[indices]
umap_emb_subsampled = umap_emb[indices]
tsne_emb_subsampled = tsne_emb[indices]
time_subsampled = time[indices]

Found 1454 NaN indices in time, dropping them from all embeddings.


In [ ]:
# compute correlation between embedded distance and time for each embedding
def compute_correlation(emb, time):
    # compute pairwise distances
    emb_dist = pdist(emb)
    assert len(emb_dist.shape) == 1, "Distance matrix shape mismatch"
    time_dist = np.abs(time - time[:, np.newaxis])
    time_dist = squareform(time_dist)
    assert len(time_dist.shape) == 1, "Time distance matrix shape mismatch"
    
    # compute spearman correlation
    spearman_corr, _ = spearmanr(emb_dist, time_dist)
    return spearman_corr

# compute correlations for each embedding
embedor_spearman = compute_correlation(embedor_emb_subsampled, time_subsampled)
print(f"Embedor Spearman: {embedor_spearman}")
umap_spearman = compute_correlation(umap_emb_subsampled, time_subsampled)
print(f"UMAP Spearman: {umap_spearman}")
tsne_spearman = compute_correlation(tsne_emb_subsampled, time_subsampled)
print(f"t-SNE Spearman: {tsne_spearman}")
# experiment with full dataset is on the server due to memory constraints, so we don't compute it here

Embedor Spearman: 0.6239895361548156
UMAP Spearman: 0.39697209626705526
t-SNE Spearman: 0.39516374880307664
